# EdgeSat AI / QAC-Net — Track 3 Colab run

This notebook runs the shared EuroSAT subset using the participant handout's exact seed-42 selection and 80/20 train/test split. It trains clean-only and noise-augmented models, evaluates both on the same untouched 400-image test set, and packages the results.

**Before running:** in Colab choose **Runtime → Change runtime type → GPU** if a GPU is available. CPU works but training takes longer. EuroSAT has land-cover labels only; the quality head detects synthetic corruption, not real clouds.

## 1. Upload the source bundle

Upload `EdgeSatAI_Submission.zip` from the downloaded project folder when the file picker opens.

In [ ]:
from google.colab import files
from pathlib import Path
import io, os, zipfile

uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
    archive.extractall('/content')

project = Path('/content/submission')
assert (project / 'run.py').exists(), 'The uploaded ZIP should contain submission/run.py'
os.chdir(project)
print('Project folder:', project)

## 2. Check the runtime

The training script selects CUDA automatically when Colab provides a GPU. It uses the same fixed split for both experiments.

In [ ]:
import torch, torchvision, sklearn
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU runtime detected; training will run on CPU.')

## 3. Train the clean-only model

The first run downloads EuroSAT. Both runs use the same selected 2,000 images and the same official 1,600/400 split. A 160-image validation slice is carved from the training pool for checkpoint selection; the official test set remains untouched.

In [ ]:
!python run.py --download --mode clean --epochs 80 --width 48 --batch-size 64 --output outputs/official_80_20/clean

## 4. Train the noise-augmented model

The test settings match the handout example: Gaussian sigma 0.15, salt-and-pepper severity 0.15, and one RGB channel dropped across each test batch.

In [ ]:
!python run.py --mode noise --epochs 80 --width 48 --batch-size 64 --output outputs/official_80_20/noise

## 5. Compare the held-out results

In [ ]:
import json
import pandas as pd
from pathlib import Path

base = Path('outputs/official_80_20')
rows = []
for mode in ('clean', 'noise'):
    result = json.loads((base / mode / 'results.json').read_text())
    row = {
        'training': mode,
        'clean_macro_f1': result['clean']['macro_f1'],
        'gaussian_macro_f1': result['noise']['gaussian']['macro_f1'],
        'gaussian_drop_pp': result['noise']['gaussian']['drop_percentage_points'],
        'salt_pepper_macro_f1': result['noise']['salt_pepper']['macro_f1'],
        'salt_pepper_drop_pp': result['noise']['salt_pepper']['drop_percentage_points'],
        'channel_dropout_macro_f1': result['noise']['channel_dropout']['macro_f1'],
        'channel_dropout_drop_pp': result['noise']['channel_dropout']['drop_percentage_points'],
        'torchscript_size_mb': result['deployment']['torchscript_size_mb'],
        'cpu_latency_ms': result['deployment']['cpu_latency_ms_per_tile'],
    }
    rows.append(row)
display(pd.DataFrame(rows).round(4))

## 6. Download the reproducibility artifacts

The archive contains both checkpoints, TorchScript exports, metrics, and the exact split manifest. Colab's CPU latency is a cloud VM measurement; it is not a flight-computer benchmark.

In [ ]:
import shutil
from google.colab import files

shutil.copy('data/eurosat/edgesat_split.json', base / 'split_manifest.json')
archive_path = shutil.make_archive('/content/EdgeSatAI_Colab_Results', 'zip', root_dir=base)
files.download(archive_path)

## Prediction demo

After training, this cell runs the noise-augmented exported model on a EuroSAT sample. Replace the image path with another RGB satellite image to try your own tile. The quality head is trained on synthetic corruption labels and does not detect clouds.

In [ ]:
!python predict.py --model outputs/official_80_20/noise/qacnet_torchscript.pt --classes model_config.json --image data/eurosat/eurosat/2750/Pasture/Pasture_509.jpg